# Improved Central Limit Theorem and Bootstrap Approximations for Linear Stochastic Approximation

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from scipy.stats import norm

# Data

In [2]:
def generate_data(n_trajs, n, theta_true, d=5, sigma=0.02, seed=2345):
    rng = np.random.default_rng(seed)
    all_data = []
    for i_traj in tqdm(range(n_trajs)):
        X = rng.uniform(-1, 1, size=(n, d))
        y = X @ theta_true + rng.normal(0, sigma, size=n)
        all_data.append(
            (X, y)
        )
    return all_data

In [3]:
n = 100000 + 1
d = 5
K = 256
theta_true = np.array([1, 1, -0.1, -0.1, -0.1])

In [4]:
n_trajs = 1024
all_data = generate_data(n_trajs, n, theta_true)

  0%|          | 0/1024 [00:00<?, ?it/s]

# Bootstrap

In [5]:
def get_boot_alphas(alphas, k, seed=1111, a=2, b=2):
    rng = np.random.default_rng(seed)
    w = rng.beta(a, b, size=(k, len(alphas)))

    mean_w = a / (a + b)
    var_w = (a * b) / ((a + b) ** 2 * (a + b + 1))
    std_w = np.sqrt(var_w)

    alphas_boot = alphas * (1 + (w - mean_w) / std_w)

    return alphas_boot

class LSA:
    def __init__(self, params):
        self.n = params['n']
        self.d = params['d']
        self.gamma = params['gamma']
        self.alphas = params['c0'] / (params['k0'] + np.arange(1, self.n + 1)) ** self.gamma

        self.A = {}
        self.b = {}

        self.K = params['K']

        self.theta_bar_save_every = dict()

        self.theta_history = None
        self.theta_bar_history = None

    def fit(self, data, theta_zero, save_every=10000, burn_in=100, save_all_history=False):
        self.A = data['A']
        self.b = data['b']

        self.theta_bar_save_every = self.count_traj(self.alphas, theta_zero=theta_zero, save_every=save_every, burn_in=burn_in, save_all_history=save_all_history)

    def count_traj(self, alphas, regime='real_world', save_every=10000):
        pass

    def get_boot_sample(self, theta_zero=None, seed=1234, save_every=10000, burn_in=100):
        alphas_boot = get_boot_alphas(self.alphas, self.K, seed)
        theta_boot = np.zeros((self.K, self.d)) + theta_zero
        return self.count_traj(alphas_boot, regime='bootstrap', theta_zero=theta_boot, save_every=save_every, burn_in=burn_in)

class LinRegLSA(LSA):
    def count_traj(self, alphas, regime='real_world', theta_zero=None, save_every=10000, burn_in=100, save_all_history=False):
        if theta_zero is None:
            theta_zero = np.zeros(self.d)
        theta = np.copy(theta_zero)
        theta_mean = np.zeros_like(theta_zero)

        theta_bar_save_every = dict()

        len_from_burn_in = 0

        if regime == 'real_world' and save_all_history:
            self.theta_history = np.zeros((self.n, self.d))
            self.theta_history[0] = theta
            self.theta_bar_history = np.zeros((self.n, self.d))
            self.theta_bar_history[0] = theta

        for i in range(1, self.n):
            if regime == 'real_world':
                theta = theta - 2 * alphas[i] * self.A[i] * (self.A[i] @ theta - self.b[i])
                if save_all_history:
                    self.theta_history[i] = theta
                    self.theta_bar_history[i] = (i * self.theta_bar_history[i - 1] + theta) / (i + 1)
            else:
                pred = theta @ self.A[i]
                error = pred - self.b[i]
                grad = error[:, None] * self.A[i]
                grad *= alphas[:, i][:, None]
                theta = theta - 2 * grad

            if i >= burn_in:
                theta_mean = (len_from_burn_in * theta_mean + theta) / (len_from_burn_in + 1)
                len_from_burn_in += 1

            if i % save_every == 0:
                theta_bar_save_every[i] = np.copy(theta_mean)
        return theta_bar_save_every

# Empirical Quantiles

In [ ]:
def count_coverage_proba(theta_true, all_data, params, conf=5, n_trajs=100, seed=3456, save_every=10000):
    proj = np.random.uniform(-1, 1, size=5)
    proj /= np.linalg.norm(proj)

    theta_true_proj = proj @ theta_true

    last_history_index = params['n'] // save_every * save_every
    cov_probas = dict([(step, 0) for step in range(save_every, last_history_index + 1, save_every)])

    pbar = tqdm(range(n_trajs))

    for i_traj in pbar:
        X, y = all_data[i_traj]

        data = {
            'A': X,
            'b': y
        }

        model = LinRegLSA(params)
        model.fit(data, theta_true)


        theta_bar_history = model.theta_bar_save_every
        theta_boot_bar_history = model.get_boot_sample(theta_true, seed=seed + i_traj, save_every=save_every)


        for step, theta_bar_boot_cur in theta_boot_bar_history.items():
            theta_bar_cur = theta_bar_history[step]

            theta_bar_proj = proj @ theta_bar_cur
            theta_bar_boot_proj = theta_bar_boot_cur @ proj
            boot_mean = np.mean(theta_bar_boot_proj)

            centered_quantiles = np.percentile(theta_bar_boot_proj, [conf / 2, 100 - conf / 2])

            q_low, q_high = centered_quantiles

            if theta_true_proj >= q_low and theta_true_proj <= q_high:
                cov_probas[step] += 1 / n_trajs

        min_cov = np.min(list(cov_probas.values()))
        max_cov = np.max(list(cov_probas.values()))
        argmins = sorted([step for step in cov_probas.keys() if cov_probas[step] == min_cov])
        argmaxs = sorted([step for step in cov_probas.keys() if cov_probas[step] == max_cov])
        pbar.set_postfix(min_cov=f"{min_cov * n_trajs / (i_traj + 1):.3f}",
                         max_cov=f"{max_cov * n_trajs / (i_traj + 1):.3f}",
                         argmins=f"{argmins}",
                         argmaxes=f"{argmaxs}")
    return cov_probas

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.95703125,
 20000: 0.953125,
 30000: 0.9423828125,
 40000: 0.951171875,
 50000: 0.9462890625,
 60000: 0.9501953125,
 70000: 0.9482421875,
 80000: 0.9501953125,
 90000: 0.955078125,
 100000: 0.9521484375}

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.9,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.94921875,
 20000: 0.953125,
 30000: 0.9443359375,
 40000: 0.947265625,
 50000: 0.9365234375,
 60000: 0.94921875,
 70000: 0.94140625,
 80000: 0.947265625,
 90000: 0.955078125,
 100000: 0.953125}

In [ ]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.9541015625,
 20000: 0.951171875,
 30000: 0.955078125,
 40000: 0.953125,
 50000: 0.951171875,
 60000: 0.951171875,
 70000: 0.9521484375,
 80000: 0.947265625,
 90000: 0.947265625,
 100000: 0.939453125}

# Standard Deviation-Based Confidence Intervals

In [13]:
def count_coverage_proba_normal(theta_true, all_data, params, conf=5, n_trajs=100, seed=3456, save_every=10000,
                                burn_in=100):
    proj = np.random.uniform(-1, 1, size=5)
    proj /= np.linalg.norm(proj)
    theta_true_proj = proj @ theta_true

    last_history_index = params['n'] // save_every * save_every
    cov_probas = dict([(step, 0) for step in range(save_every, last_history_index + 1, save_every)])

    pbar = tqdm(range(n_trajs))

    for i_traj in pbar:
        X, y = all_data[i_traj]

        data = {
            'A': X,
            'b': y
        }

        model = LinRegLSA(params)
        model.fit(data, theta_true, burn_in=burn_in)


        theta_bar_history = model.theta_bar_save_every
        theta_boot_bar_history = model.get_boot_sample(theta_true, seed=seed + i_traj, save_every=save_every, burn_in=burn_in)


        for step, theta_bar_boot_cur in theta_boot_bar_history.items():
            theta_bar_cur = theta_bar_history[step]

            m = step - burn_in

            theta_bar_proj = proj @ theta_bar_cur
            theta_bar_boot_proj = np.sqrt(m) * (theta_bar_boot_cur @ proj - theta_bar_proj)

            var = np.var(theta_bar_boot_proj)
            std = np.sqrt(var)

            alpha = conf / 100
            z = norm.ppf(1 - alpha/2)

            q_low = theta_bar_proj - z * std / np.sqrt(m)
            q_high = theta_bar_proj + z * std / np.sqrt(m)

            if q_low <= theta_true_proj <= q_high:
                cov_probas[step] += 1 / n_trajs
        min_cov = np.min(list(cov_probas.values()))
        max_cov = np.max(list(cov_probas.values()))
        argmins = sorted([step for step in cov_probas.keys() if cov_probas[step] == min_cov])
        argmaxs = sorted([step for step in cov_probas.keys() if cov_probas[step] == max_cov])
        pbar.set_postfix(min_cov=f"{min_cov * n_trajs / (i_traj + 1):.3f}",
                         max_cov=f"{max_cov * n_trajs / (i_traj + 1):.3f}",
                         argmins=f"{argmins}",
                         argmaxes=f"{argmaxs}")
    return cov_probas

In [15]:
params = {
    'n': 50000 + 1,
    'd': 5,
    'K': 256,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba_normal(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.9619140625,
 20000: 0.958984375,
 30000: 0.9521484375,
 40000: 0.94921875,
 50000: 0.947265625}

In [16]:
params = {
    'n': 50000 + 1,
    'd': 5,
    'K': 256,
    'gamma': 0.9,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba_normal(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.9609375,
 20000: 0.96484375,
 30000: 0.9580078125,
 40000: 0.953125,
 50000: 0.9609375}

In [17]:
params = {
    'n': 50000 + 1,
    'd': 5,
    'K': 256,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

count_coverage_proba_normal(theta_true, all_data, params, n_trajs=n_trajs)

  0%|          | 0/1024 [00:00<?, ?it/s]

{10000: 0.9462890625,
 20000: 0.9541015625,
 30000: 0.955078125,
 40000: 0.947265625,
 50000: 0.94921875}

# Overlapping Batch Mean Estimator

In [6]:
def batch_mean(theta_hat_history, b_n):
    N_iters, N_s = theta_hat_history.shape
    BM_V = np.empty((N_iters - b_n + 1, N_s), dtype=theta_hat_history.dtype)

    cur_bm = theta_hat_history[0:b_n, :].mean(axis=0)
    for i in range(N_iters - b_n):
        BM_V[i, :] = cur_bm
        cur_bm += (theta_hat_history[b_n + i, :] - theta_hat_history[i, :]) / b_n
    BM_V[-1, :] = cur_bm
    return BM_V

def cov_prob_3(all_data, params, conf, b_n, n_trajs, i):
    rng = np.random.default_rng(12345)
    u = rng.normal(size=params['d'])
    u /= np.linalg.norm(u)
    theta_in_interval = np.zeros(n_trajs, dtype=bool)
    for i_traj in tqdm(range(n_trajs), desc=f"Computing coverage for b_n = {b_n}"):
        X, y = all_data[i_traj]

        data = {
            'A': X,
            'b': y
        }

        model = LinRegLSA(params)
        model.fit(data, theta_true, save_all_history=True)

        theta_hat_history = model.theta_history
        theta_bar_history = model.theta_bar_history

        BM_V = batch_mean(theta_hat_history, b_n)

        hat_sigma =  (((BM_V - theta_bar_history[-1, :]) @ u) ** 2).sum() * (b_n / (50001-b_n))
        std = np.sqrt(hat_sigma)
        alpha = 1 - conf / 100
        z = norm.ppf(1 - alpha / 2)

        lower = theta_bar_history[(i + 1) * 10000, :] @ u - z * std / np.sqrt((i + 1) * 10000 + 1)
        upper = theta_bar_history[(i + 1) * 10000, :] @ u + z * std / np.sqrt((i + 1) * 10000 + 1)

        true_proj = theta_true @ u
        theta_in_interval[i_traj] = int((lower <= true_proj <= upper))
    coverage = np.mean(theta_in_interval)
    return coverage

In [8]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.85,
    'c0': 200,
    'k0': 20000,
}

for b_n in range(1000, 1600 + 1, 200):
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 1000:   0%|          | 0/1024 [00:00<?, ?it/s]

1000 0.9912109375


Computing coverage for b_n = 1200:   0%|          | 0/1024 [00:00<?, ?it/s]

1200 0.9931640625


Computing coverage for b_n = 1400:   0%|          | 0/1024 [00:00<?, ?it/s]

1400 0.9931640625


Computing coverage for b_n = 1600:   0%|          | 0/1024 [00:00<?, ?it/s]

1600 0.9951171875


In [9]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.9,
    'c0': 200,
    'k0': 20000,
}

for b_n in range(1000, 1600 + 1, 200):
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 1000:   0%|          | 0/1024 [00:00<?, ?it/s]

1000 0.9892578125


Computing coverage for b_n = 1200:   0%|          | 0/1024 [00:00<?, ?it/s]

1200 0.9912109375


Computing coverage for b_n = 1400:   0%|          | 0/1024 [00:00<?, ?it/s]

1400 0.9921875


Computing coverage for b_n = 1600:   0%|          | 0/1024 [00:00<?, ?it/s]

1600 0.994140625


In [10]:
params = {
    'n': n,
    'd': d,
    'K': K,
    'gamma': 0.95,
    'c0': 200,
    'k0': 20000,
}

for b_n in range(1000, 1600 + 1, 200):
    print(b_n, cov_prob_3(all_data, params, 95, b_n, n_trajs, 4))

Computing coverage for b_n = 1000:   0%|          | 0/1024 [00:00<?, ?it/s]

1000 0.98046875


Computing coverage for b_n = 1200:   0%|          | 0/1024 [00:00<?, ?it/s]

1200 0.982421875


Computing coverage for b_n = 1400:   0%|          | 0/1024 [00:00<?, ?it/s]

1400 0.984375


Computing coverage for b_n = 1600:   0%|          | 0/1024 [00:00<?, ?it/s]

1600 0.98828125
